# Лабораторная работа 1. Анализ тональности на основе BERT

### Задание 1. Получение оценок качества с классической моделью BERT
В этом задании требуется получить оценки качества на обучающих данных Kaggle (https://www.kaggle.com/c/sentiment-analysis-in-russian/data) с применением модели `RuBERT` на основе отложенной выборки или кросс-валидации.  

При выполнении задания можно воспользоваться предоставленным ноутбуком (`huggingface_bert_finetuning`).  
Подберите гиперпараметры (по крайней мере, количество эпох).  

Выведите результаты и время построения моделей в удобном табличном виде, а также в виде графиков.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install transformers --upgrade

In [ ]:
# После импорта библиотек добавьте:
print("Проверка версий библиотек...")
import transformers
print(f"Transformers version: {transformers.__version__}")

# Если версия очень старая, можно принудительно обновить:
# !pip install transformers==4.36.0

In [ ]:
# Импортируем необходимые библиотеки
import pandas as pd
import json
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support
import torch
from transformers import BertForSequenceClassification, BertTokenizer, BertConfig, Trainer, TrainingArguments
from datasets import Dataset
import time
import random
from torch.nn.functional import softmax

# Устанавливаем seed для воспроизводимости
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

# Загрузка данных Kaggle
train_path = '/content/drive/MyDrive/DL/DL_labs/dl_lab_1/train.json'
test_path = '/content/drive/MyDrive/DL/DL_labs/dl_lab_1/test.json'
sample_path = '/content/drive/MyDrive/DL/DL_labs/dl_lab_1/sample.csv'

# Загружаем тренировочные данные
with open(train_path, 'r', encoding='utf-8') as f:
    train_data = json.load(f)

# Загружаем тестовые данные
with open(test_path, 'r', encoding='utf-8') as f:
    test_data = json.load(f)

# Преобразуем в DataFrame
train_df = pd.DataFrame(train_data)
test_df = pd.DataFrame(test_data)

##Предобработка данных

In [ ]:
# Проверим распределение классов
print("Распределение классов в тренировочных данных:")
print(train_df['sentiment'].value_counts())
print(f"\nВсего классов: {train_df['sentiment'].nunique()}")
print(f"Уникальные значения: {train_df['sentiment'].unique()}")

# Создаем маппинг меток для BERT
unique_classes = sorted(train_df['sentiment'].unique())
label_to_id = {label: idx for idx, label in enumerate(unique_classes)}
id_to_label = {idx: label for label, idx in label_to_id.items()}

num_classes = len(unique_classes)  # ← ВАЖНО: определяем num_classes здесь!

print(f"Количество классов: {num_classes}")
print(f"Маппинг меток: {label_to_id}")

# Применяем маппинг к тренировочным данным
train_df['label'] = train_df['sentiment'].map(label_to_id)

##Инциализация модели и токенизация

In [ ]:
# Проверяем доступность GPU
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"\nИспользуемое устройство: {device}")

# Выбираем модель
PRE_TRAINED_MODEL_NAME = 'blanchefort/rubert-base-cased-sentiment'

# Инициализируем токенизатор
tokenizer = BertTokenizer.from_pretrained(PRE_TRAINED_MODEL_NAME)

# Анализируем длину текстов
def analyze_text_lengths(df, tokenizer, column='text'):
    token_lengths = []
    for text in df[column]:
        tokens = tokenizer.encode(text, truncation=False)
        token_lengths.append(len(tokens))

    print(f"Минимальная длина: {min(token_lengths)} токенов")
    print(f"Максимальная длина: {max(token_lengths)} токенов")
    print(f"Средняя длина: {np.mean(token_lengths):.2f} токенов")
    print(f"Медианная длина: {np.median(token_lengths):.2f} токенов")

    return token_lengths

print("\nАнализ длины текстов в тренировочных данных:")
train_lengths = analyze_text_lengths(train_df, tokenizer)

# Определяем MAX_LEN
percentile_95 = np.percentile(train_lengths, 95)
MAX_LEN = min(256, int(percentile_95 * 1.1))
print(f"\n95-й перцентиль длины: {percentile_95:.2f}")
print(f"Выбранный MAX_LEN: {MAX_LEN}")

##Подготовка данных для BERT

In [ ]:
# Разделяем данные на тренировочные и валидационные
train_data, val_data = train_test_split(
    train_df,
    test_size=0.1,
    random_state=seed,
    stratify=train_df['label']
)

print(f"\nРазмер тренировочного набора: {len(train_data)}")
print(f"Размер валидационного набора: {len(val_data)}")

# Функция для токенизации
def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        max_length=MAX_LEN,
        padding='max_length',
        truncation=True,
        return_tensors=None
    )

# Создаем Dataset объекты
train_dataset = Dataset.from_pandas(train_data[['text', 'label']])
val_dataset = Dataset.from_pandas(val_data[['text', 'label']])

# Применяем токенизацию
print("Токенизация данных...")
train_dataset = train_dataset.map(tokenize_function, batched=True, batch_size=32)
val_dataset = val_dataset.map(tokenize_function, batched=True, batch_size=32)

# Устанавливаем формат для PyTorch
train_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
val_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])

# Функция для вычисления метрик
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average=None, zero_division=0
    )
    acc = accuracy_score(labels, preds)

    macro_precision = np.mean(precision)
    macro_recall = np.mean(recall)
    macro_f1 = np.mean(f1)

    results = {
        'accuracy': acc,
        'macro_precision': macro_precision,
        'macro_recall': macro_recall,
        'macro_f1': macro_f1,
    }

    # Добавляем метрики по классам
    for i in range(len(precision)):
        results[f'precision_class_{i}'] = precision[i]
        results[f'recall_class_{i}'] = recall[i]
        results[f'f1_class_{i}'] = f1[i]

    return results

##Обучение модели с подбором гиперпараметров

In [ ]:
def train_model(num_epochs, learning_rate, batch_size, model_name_suffix=""):
    """Обучение модели с заданными гиперпараметрами"""

    timestamp = int(time.time())
    output_dir = f"./results_{model_name_suffix}_{timestamp}"

    # Определяем версию transformers для совместимости
    try:
        import transformers
        transformers_version = transformers.__version__
        print(f"Версия transformers: {transformers_version}")
    except:
        transformers_version = "4.0.0"

    # Параметры обучения с учетом совместимости версий
    training_args_dict = {
        'output_dir': output_dir,
        'num_train_epochs': num_epochs,
        'per_device_train_batch_size': batch_size,
        'per_device_eval_batch_size': batch_size * 2,
        'learning_rate': learning_rate,
        'weight_decay': 0.01,
        'warmup_steps': 100,
        'logging_dir': f'./logs_{model_name_suffix}_{timestamp}',
        'logging_steps': 50,
        'load_best_model_at_end': True,
        'seed': seed,
        'report_to': 'none',
        'save_strategy': 'epoch',
        'eval_strategy': 'epoch',
        'save_total_limit': 2,
        'metric_for_best_model': 'eval_accuracy',
        'greater_is_better': True,
    }

    # Для обратной совместимости
    if 'evaluation_strategy' in TrainingArguments.__init__.__code__.co_varnames:
        training_args_dict['evaluation_strategy'] = 'epoch'
    if 'eval_strategy' in TrainingArguments.__init__.__code__.co_varnames:
        training_args_dict['eval_strategy'] = 'epoch'

    training_args = TrainingArguments(**training_args_dict)

    # Загружаем модель
    print(f"Загружаем модель {PRE_TRAINED_MODEL_NAME}...")

    try:
        model = BertForSequenceClassification.from_pretrained(
            PRE_TRAINED_MODEL_NAME,
            num_labels=num_classes,
            ignore_mismatched_sizes=True
        )
    except Exception as e:
        print(f"Ошибка при загрузке модели: {e}")
        print("Пробуем альтернативный способ...")
        config = BertConfig.from_pretrained(PRE_TRAINED_MODEL_NAME)
        config.num_labels = num_classes
        model = BertForSequenceClassification(config)

    model.to(device)

    # Создаем Trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics
    )

    print(f"\n{'='*60}")
    print(f"Обучение модели с параметрами:")
    print(f"  Количество эпох: {num_epochs}")
    print(f"  Learning rate: {learning_rate}")
    print(f"  Batch size: {batch_size}")
    print(f"{'='*60}\n")

    start_time = time.time()
    train_result = trainer.train()
    training_time = time.time() - start_time

    eval_result = trainer.evaluate()

    print(f"\nРезультаты обучения:")
    print(f"  Время обучения: {training_time:.2f} секунд")
    print(f"  Финальные потери: {train_result.metrics['train_loss']:.4f}")
    print(f"\nРезультаты на валидации:")
    print(f"  Accuracy: {eval_result['eval_accuracy']:.4f}")
    print(f"  Macro F1: {eval_result['eval_macro_f1']:.4f}")

    return trainer, eval_result, training_time


# Также обновим функцию compute_metrics для лучшей обработки:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average=None, zero_division=0
    )
    acc = accuracy_score(labels, preds)

    macro_precision = np.mean(precision)
    macro_recall = np.mean(recall)
    macro_f1 = np.mean(f1)

    results = {
        'accuracy': acc,
        'macro_precision': macro_precision,
        'macro_recall': macro_recall,
        'macro_f1': macro_f1,
    }

    # Добавляем метрики по классам
    for i in range(len(precision)):
        results[f'precision_class_{i}'] = precision[i]
        results[f'recall_class_{i}'] = recall[i]
        results[f'f1_class_{i}'] = f1[i]

    return results

## Экспериментируем с разным количеством эпох

In [ ]:
# Импортируем дополнительно os для работы с путями
import os

# Список эпох для инкрементального обучения
epochs_sequence = [2, 3, 4, 5]  # 2 -> +1 -> +1 -> +1

# Переменная для хранения пути к последней обученной модели
previous_model_path = None
results = []  # Список для сохранения результатов

print("НАЧАЛО ИНКРЕМЕНТАЛЬНОГО ОБУЧЕНИЯ")
print("="*60)

for i, total_epochs in enumerate(epochs_sequence):
    print(f"\n{'#'*60}")
    print(f"ЭТАП {i+1}: Обучение до {total_epochs} эпох")
    if previous_model_path:
        print(f"Продолжаем с модели: {previous_model_path}")
    else:
        print(f"Начинаем обучение с нуля")
    print(f"{'#'*60}")

    # Определяем количество эпох для ДАННОГО этапа обучения
    if previous_model_path:
        # Если есть предыдущая модель, дообучаем ее 1 эпоху
        epochs_to_train = 1
    else:
        # Иначе обучаем с нуля начальное количество эпох
        epochs_to_train = total_epochs

    timestamp = int(time.time())
    output_dir = f"./incremental_model_epochs_{total_epochs}_{timestamp}"

    # Параметры обучения для текущего этапа
    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=epochs_to_train,  # Обучаем 1 эпоху на инкрементальных этапах
        per_device_train_batch_size=32,
        per_device_eval_batch_size=64,
        learning_rate=2e-5,
        weight_decay=0.01,
        warmup_steps=100,
        logging_dir=f'./logs_incremental_{total_epochs}_{timestamp}',
        logging_steps=50,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        seed=seed,
        report_to='none',
        save_total_limit=1
    )

    # Загружаем модель: либо предыдущую, либо новую
    print(f"Загружаем модель...")
    try:
        if previous_model_path and os.path.exists(previous_model_path):
            # Продолжаем обучение с предыдущей модели
            model = BertForSequenceClassification.from_pretrained(previous_model_path)
            print(f"Загружена ранее обученная модель из {previous_model_path}")
        else:
            # Начинаем обучение с нуля (только на первом этапе)
            model = BertForSequenceClassification.from_pretrained(
                PRE_TRAINED_MODEL_NAME,
                num_labels=num_classes,
                ignore_mismatched_sizes=True
            )
            print(f"Загружена предобученная модель {PRE_TRAINED_MODEL_NAME}")
    except Exception as e:
        print(f"Ошибка при загрузке модели: {e}")
        # Альтернативный способ загрузки
        config = BertConfig.from_pretrained(PRE_TRAINED_MODEL_NAME)
        config.num_labels = num_classes
        model = BertForSequenceClassification.from_pretrained(
            PRE_TRAINED_MODEL_NAME,
            config=config,
            ignore_mismatched_sizes=True
        )

    model.to(device)

    # Создаем Trainer для текущего этапа
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics
    )

    # Обучаем модель
    print(f"\nНачало обучения на {epochs_to_train} эпоху(и)...")
    start_time = time.time()

    if previous_model_path:
        # При продолжении обучения используем train() без сброса
        train_result = trainer.train(resume_from_checkpoint=False)
    else:
        # При обучении с нуля
        train_result = trainer.train()

    training_time = time.time() - start_time

    # Сохраняем модель для использования на следующем этапе
    trainer.save_model(output_dir)
    previous_model_path = output_dir  # Запоминаем путь для следующего этапа

    # Оцениваем модель
    eval_result = trainer.evaluate()

    # Сохраняем результаты
    results.append({
        'total_epochs': total_epochs,
        'incremental_epochs': epochs_to_train,
        'accuracy': eval_result['eval_accuracy'],
        'macro_f1': eval_result['eval_macro_f1'],
        'macro_precision': eval_result['eval_macro_precision'],
        'macro_recall': eval_result['eval_macro_recall'],
        'training_time_this_stage': training_time,
        'model_path': output_dir
    })

    # Выводим промежуточные результаты
    cumulative_time = sum(r['training_time_this_stage'] for r in results)
    print(f"\nИтоги этапа {total_epochs} эпох:")
    print(f"  Время на этом этапе: {training_time:.2f} сек")
    print(f"  Общее время обучения: {cumulative_time:.2f} сек")
    print(f"  Accuracy: {eval_result['eval_accuracy']:.4f}")
    print(f"  Macro F1: {eval_result['eval_macro_f1']:.4f}")
    print(f"  Модель сохранена в: {output_dir}")

# Создаем сводную таблицу результатов
print("\n" + "="*80)
print("ИТОГИ ИНКРЕМЕНТАЛЬНОГО ОБУЧЕНИЯ:")
print("="*80)

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

# Визуализация прогресса обучения
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# График точности
axes[0, 0].plot(results_df['total_epochs'], results_df['accuracy'], marker='o', linewidth=2)
axes[0, 0].set_title('Accuracy на валидации', fontsize=14)
axes[0, 0].set_xlabel('Всего эпох', fontsize=12)
axes[0, 0].set_ylabel('Accuracy', fontsize=12)
axes[0, 0].grid(True, alpha=0.3)

# График F1-score
axes[0, 1].plot(results_df['total_epochs'], results_df['macro_f1'], marker='s', linewidth=2, color='green')
axes[0, 1].set_title('Macro F1 на валидации', fontsize=14)
axes[0, 1].set_xlabel('Всего эпох', fontsize=12)
axes[0, 1].set_ylabel('F1-score', fontsize=12)
axes[0, 1].grid(True, alpha=0.3)

# График времени обучения (накопительный)
cumulative_time = np.cumsum(results_df['training_time_this_stage'])
axes[1, 0].bar(results_df['total_epochs'], results_df['training_time_this_stage'], alpha=0.7, label='На этапе')
axes[1, 0].plot(results_df['total_epochs'], cumulative_time, marker='o', color='red', linewidth=2, label='Общее время')
axes[1, 0].set_title('Время обучения', fontsize=14)
axes[1, 0].set_xlabel('Всего эпох', fontsize=12)
axes[1, 0].set_ylabel('Время (сек)', fontsize=12)
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# График Precision и Recall
axes[1, 1].plot(results_df['total_epochs'], results_df['macro_precision'], marker='^', linewidth=2, label='Precision')
axes[1, 1].plot(results_df['total_epochs'], results_df['macro_recall'], marker='v', linewidth=2, label='Recall')
axes[1, 1].set_title('Precision и Recall (macro)', fontsize=14)
axes[1, 1].set_xlabel('Всего эпох', fontsize=12)
axes[1, 1].set_ylabel('Значение', fontsize=12)
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nИнкрементальное обучение завершено!")
print(f"Финальная модель сохранена в: {previous_model_path}")

### Задание 2. Использование BERT как модель для формирования векторных представлений текстов
Основная функция BERT-подобных моделей – формирование контекстно-ориентированных векторных представлений текстов.  
В данном задании нужно будет сделать следующее:
1) обучить RuBERT (как в предыдущем задании);
1) получить из него векторные представления обучающих текстов (например, вектор токена `[CLS]` или среднее векторов всех токенов на последнем слое (см. варианты [здесь](https://mccormickml.com/2019/05/14/BERT-word-embeddings-tutorial/#35-pooling-strategy--layer-choice));
1) применить традиционные модели машинного обучения (логистическая регрессия, SVM, градиентный бустинг и т.п.). Не забывайте про подбор гиперапараметров;
1) получить оценки качества – на таком же разбиении, как в предыдущем задании.

Выведите результаты и время построения моделей в удобном табличном виде, а также в виде графиков.

### Задание 3. Оценка качества классификации на Kaggle
Выберите лучшую модель, обучите её на всем обучающем корпусе, получите предсказания для тестовых данных и отправьте результаты на Kaggle.  

Сравните полученные результаты с результатами на Kaggle: https://www.kaggle.com/c/sentiment-analysis-in-russian/leaderboard.

Выведите кроме `macro F1-score` следующие метрики для отложенной выборки:
- `F1-score` по каждому классу;
- `Precision` и `Recall` по каждому классу;
- `Confusion matrix` по каждому классу и для всей выборки.

Сделайте выводы в целом по своим исследованиям – приведите в одной таблице несколько лучших моделей с указанием параметров и времени работы.

In [ ]:
# Ваш код здесь